In [ ]:
import os, glob
import numpy as np # linear algebra
import pandas as pd 

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from typing import TypedDict, Annotated, Optional, List, Literal
from pydantic import BaseModel, EmailStr, Field

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnableParallel
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain_huggingface import HuggingFaceEmbeddings

##### 0- Model & Path

In [ ]:
Amodel = ChatAnthropic(
    model='claude-sonnet-4-5-20250929',
    anthropic_api_key="")

In [ ]:
DATA_DIR = "./pakistan_laws"            # put your files here
PERSIST_DIR = "./chroma_db"    # vector DB folder

##### 1- Loader

In [ ]:
pdf_paths = glob.glob(os.path.join(DATA_DIR, "**/*.pdf"), recursive=True)
pdf_docs = []
bad_pdfs = []
for path in pdf_paths:
    try:
        loader = PyPDFLoader(path)
        pdf_docs.extend(loader.load())
    except Exception as e:
        bad_pdfs.append((path, str(e)))
print(f" Loaded PDF docs: {len(pdf_docs)}")

##### 2- Splitter

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)
splits = splitter.split_documents(pdf_docs)
print(len(splits))
splits[:10]

##### 3- Vector

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    show_progress=True,
    model_kwargs={'trust_remote_code': True}
)

##### 4- Vector DB

In [ ]:
# vectordb = FAISS.from_documents(
#     documents=splits,
#     embedding=embeddings
# )
# vectordb.save_local("./faiss_index")

In [ ]:
# vectordb = Chroma.from_documents(
#     documents=splits,
#     embedding=embeddings,
#     persist_directory="./chroma_db",
#     collection_name="rag_notebook"
# )

In [ ]:
vectordb = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings,
    collection_name="rag_notebook"  # Match the name you used!
)

print(f"Loaded {vectordb._collection.count()} vectors")

In [ ]:
data = vectordb.get(limit=5)
data


##### 5- Retriver

In [ ]:
retriever = vectordb.as_retriever(search_kwargs={"k": 4})

In [ ]:
query = "I want to inqure what law say about the fraud calls and fraud banking"

In [ ]:
docs = retriever.invoke(query)
docs

###### Prompt

In [ ]:
prompt = PromptTemplate.from_template("""
        Answer the question ONLY from the provided context and summarise and rephrase in easy words.
        
        Question:
        {question}
        
        Context:
        {context}
        
        If answer not in context, say:
        "I don't know based on the documents."
        """)

llm = Amodel
parser = StrOutputParser()

##### 6- Brain

In [ ]:
def rag_query(question: str):
    docs = retriever.invoke(question)
    context = "\n\n".join([f"[Doc {i+1}] {d.page_content}" for i, d in enumerate(docs)])
    # print(context,"\n\n")
    chain = prompt | llm | parser
    answer = chain.invoke({"question": question, "context": context})
    return answer, docs

##### Final Test

In [ ]:
answer, sources = rag_query(query)

print("ANSWER:\n")
print(answer)